<a href="https://colab.research.google.com/github/craljimenez/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/blob/main/clases/semana10-clase20-regresion-lineal-python/notebooks/lab-regresion-lineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: regresión lineal con Python
### Semana 10 · Clase 20 · Ruta Python
**Curso:** Análisis y Visualización de Datos para la Toma de Decisiones (Esumer)

En la Clase 19 viste la idea: una **variable objetivo** que quieres predecir, unas **variables explicativas**, un reparto entre **entrenamiento y prueba** y una **línea base** contra la cual comparar. Hoy lo haces de verdad, con el modelo más sencillo y más usado: la **regresión lineal**.

**El caso.** Capital Bikeshare alquila bicicletas en Washington D. C. Cada noche debe decidir cuántas bicicletas y cuánto personal tener listos para el día siguiente. Tienes dos años de datos reales, día por día (2011 y 2012), con el clima de cada día. La pregunta:

> **¿Cuántas bicicletas se alquilarán en un día, según el clima y el tipo de día?**

Es un problema de **regresión**: el objetivo es un número.

**Hay dos rutas para esta práctica.** Esta es la de Python. La otra se hace en Orange, con una guía en PDF. Las dos usan los mismos datos y hacen el reparto entre entrenamiento y prueba con la misma semilla, así que **eligen los mismos días y los números deben salir iguales**: si trabajas en grupo, comparen.

**Sobre los números de este notebook:** Python escribe los decimales con punto (160.35). En el texto los escribimos con coma, como en español (160,35).

## Los datos: `bicicletas_diarias.csv`

Una fila por día, del 1 de enero de 2011 al 31 de diciembre de 2012 (731 días).

| Columna | Qué contiene |
|---|---|
| `fecha`, `anio`, `mes`, `dia_semana` | Cuándo |
| `dia_laboral` | 1 si es día laboral; 0 si es fin de semana o festivo |
| `clima` | Despejado, Nublado o con niebla, Lluvia o nieve |
| `temperatura_c`, `sensacion_termica_c` | Temperatura y sensación térmica, en grados Celsius |
| `humedad` | Humedad relativa, de 0 a 100 |
| `viento` | Velocidad del viento, en la escala del archivo original |
| `es_2012` | 1 si el día es de 2012; 0 si es de 2011 |
| `lluvia` | 1 si hubo lluvia o nieve; 0 si no |
| `alquileres` | **Objetivo:** bicicletas alquiladas ese día |

**Fuente:** *Bike Sharing Dataset* (Capital Bikeshare), UCI Machine Learning Repository, licencia CC BY 4.0. Fanaee-T, H. y Gama, J. (2013). *Event labeling combining ensemble detectors and background knowledge*. Progress in Artificial Intelligence. Las columnas se tradujeron al español y las temperaturas, la humedad y el viento se devolvieron a su escala original.

## 0. Cargar los datos

Ejecuta la celda y selecciona `bicicletas_diarias.csv`, de la carpeta `datos/` de esta clase en el repositorio del curso.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

dias = pd.read_csv("bicicletas_diarias.csv")
dias.shape

**Debes ver:** `(731, 13)`: 731 días y 13 columnas.

In [ ]:
TEAL = "#0CB3B6"
NAVY = "#262E66"
ORANGE = "#EF7C2C"
SKY = "#26A1DB"

dias.head()

## 1. Conocer los datos

Antes de modelar, la revisión de siempre (Clase 7): rangos y valores raros.

In [ ]:
dias[["temperatura_c", "humedad", "viento", "alquileres"]].describe().round(1)

**Mira la humedad mínima: 0.** Una humedad relativa de cero no ocurre en una ciudad: es un error de registro. Busca qué día es:

In [ ]:
dias[dias["humedad"] == 0]

Es un solo día (10 de marzo de 2011). Lo dejamos, porque un día entre 731 casi no mueve el modelo, pero lo anotamos: **en tu proyecto, cada decisión sobre un dato raro va en la bitácora** (Clase 12).

## 2. Mirar la relación antes de modelar

La regresión lineal supone que la relación es, más o menos, una **línea recta**. Antes de ajustar nada, míralo: ¿los días más cálidos tienen más alquileres?

In [ ]:
dias.plot.scatter(x="temperatura_c", y="alquileres", color=SKY, alpha=0.5)
plt.title("A más temperatura, más alquileres... hasta cierto punto")
plt.xlabel("Temperatura (grados Celsius)")
plt.ylabel("Bicicletas alquiladas en el día")
plt.show()

In [ ]:
dias["temperatura_c"].corr(dias["alquileres"]).round(3)

**Debes ver:** una correlación de **0,627**: relación positiva y moderada. Fíjate también en la forma de la nube: sube, pero en los días más calurosos parece bajar un poco. Lo vas a confirmar en el paso 5.

## 3. Separar entrenamiento y prueba

Como en la Clase 19, con `train_test_split`: aparta el 20% de los días, al azar, para la prueba.

- `test_size=0.2`: el 20% para la prueba y el resto para el entrenamiento.
- `random_state=42`: fija el azar, para que el reparto sea el mismo cada vez que ejecutas la celda y el mismo de tus compañeros. En Orange, el widget **Data Sampler** con "muestreo replicable" usa la misma semilla y elige **los mismos días**.

In [ ]:
from sklearn.model_selection import train_test_split

entrenamiento, prueba = train_test_split(dias, test_size=0.2, random_state=42)

len(entrenamiento), len(prueba)

**Debes ver:** `(584, 147)`. El modelo aprende **solo** con los 584 días de entrenamiento; los 147 de prueba son el examen que no ha visto. Si te salen otras cifras más adelante, revisa primero que escribiste `random_state=42`.

## 4. La línea base: predecir siempre el promedio

Antes de celebrar cualquier modelo, ¿cuánto se equivoca alguien que dice siempre el mismo número: el promedio de alquileres del entrenamiento?

Para medir el error usamos el **error absoluto medio** (MAE, por sus siglas en inglés): en promedio, ¿por cuántas bicicletas se equivoca la predicción? Está en las mismas unidades del objetivo, así que se explica fácil.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

promedio = entrenamiento["alquileres"].mean()
pred_base = [promedio] * len(prueba)

print("Promedio de entrenamiento:", round(promedio, 1))
print("MAE de la línea base:", round(mean_absolute_error(prueba["alquileres"], pred_base), 1))

**Debes ver:** promedio de **4561,0** bicicletas y un MAE de **1712,0**. Sin mirar ningún dato, nos equivocamos en unas 1700 bicicletas por día. Todo modelo tiene que hacerlo mejor que eso.

## 5. Regresión lineal simple: solo la temperatura

El modelo busca la recta que mejor sigue a los puntos:

> **alquileres = intercepto + pendiente × temperatura**

En scikit-learn, todos los modelos se usan igual: se crea, se **ajusta** con `fit` y se **predice** con `predict`.

In [ ]:
from sklearn.linear_model import LinearRegression

X_train = entrenamiento[["temperatura_c"]]
y_train = entrenamiento["alquileres"]
X_test = prueba[["temperatura_c"]]
y_test = prueba["alquileres"]

simple = LinearRegression()
simple.fit(X_train, y_train)

print("Intercepto:", round(simple.intercept_, 2))
print("Pendiente (temperatura):", round(simple.coef_[0], 2))

**Debes ver:** intercepto **1272,76** y pendiente **160,35**.

**Cómo se lee, en lenguaje de negocio:** *por cada grado más de temperatura, se alquilan en promedio unas 160 bicicletas más al día.* El intercepto sería el valor a 0 grados, pero ningún día de los datos llegó a 0 (el más frío tuvo 2,4): por eso no conviene leerlo como una predicción real.

`X_train` va entre **dos** corchetes porque el modelo espera una tabla de columnas, aunque sea una sola; `y_train` va con uno porque es una sola columna.

In [ ]:
orden = X_train.sort_values("temperatura_c")

entrenamiento.plot.scatter(x="temperatura_c", y="alquileres", color=SKY, alpha=0.5)
plt.plot(orden["temperatura_c"], simple.predict(orden), color=ORANGE, linewidth=3)
plt.title("La recta que aprendió el modelo")
plt.xlabel("Temperatura (grados Celsius)")
plt.ylabel("Bicicletas alquiladas en el día")
plt.show()

Ahora el examen: predecir los 146 días de prueba y medir el error con tres medidas.

- **MAE:** el error promedio, en bicicletas.
- **RMSE** (raíz del error cuadrático medio): también en bicicletas, pero castiga más los errores grandes.
- **R²:** qué parte de la variación de los alquileres explica el modelo. 1 es perfecto; 0 es lo mismo que predecir el promedio.

In [ ]:
pred_simple = simple.predict(X_test)

print("MAE: ", round(mean_absolute_error(y_test, pred_simple), 1))
print("RMSE:", round(mean_squared_error(y_test, pred_simple) ** 0.5, 1))
print("R²:  ", round(r2_score(y_test, pred_simple), 3))

**Debes ver:** MAE **1291,7**, RMSE **1545,9** y R² **0,404**. Mejor que la línea base (1712,0), pero todavía nos equivocamos en unas 1300 bicicletas por día: la temperatura sola explica cerca del 40% de la variación.

## 6. Los residuos: lo que el modelo no explica

El **residuo** de cada día es lo real menos lo predicho. Si el modelo es adecuado, los residuos se reparten al azar alrededor de cero, sin forma. Si tienen forma, el modelo se está perdiendo algo.

In [ ]:
entrenamiento = entrenamiento.copy()
entrenamiento["residuo"] = y_train - simple.predict(X_train)

entrenamiento.plot.scatter(x="temperatura_c", y="residuo", color=NAVY, alpha=0.5)
plt.axhline(0, color=ORANGE, linewidth=2)
plt.title("Los residuos tienen forma de arco: la relación no es recta")
plt.xlabel("Temperatura (grados Celsius)")
plt.ylabel("Residuo (real menos predicho)")
plt.show()

In [ ]:
tramos = pd.cut(entrenamiento["temperatura_c"], bins=[0, 10, 20, 30, 40])
entrenamiento.groupby(tramos, observed=True)["residuo"].agg(["mean", "count"]).round(0)

**Debes ver:** residuo promedio de **-897** entre 0 y 10 grados, **107** entre 10 y 20, **290** entre 20 y 30, y **-923** por encima de 30.

**Cómo se lee:** en los días fríos y en los muy calurosos, el modelo **predice de más** (residuo negativo); en los templados, **de menos**. Con el calor fuerte la gente deja de montar en bicicleta, y una recta no puede doblarse para capturarlo. Es un **límite del modelo lineal**: no un error tuyo.

## 7. Regresión lineal múltiple: más variables explicativas

Con lo que aprendiste en la Clase 19, agrega variables que se conocen **antes** de que pase el día (el pronóstico del clima, el calendario) y que no repiten lo que dice otra. No usamos `sensacion_termica_c`: su correlación con la temperatura es de 0,99, así que dice casi lo mismo.

In [ ]:
columnas = ["temperatura_c", "humedad", "viento", "es_2012", "dia_laboral", "lluvia"]

multiple = LinearRegression()
multiple.fit(entrenamiento[columnas], y_train)

print("Intercepto:", round(multiple.intercept_, 2))
pd.Series(multiple.coef_, index=columnas).round(2)

**Debes ver:** intercepto **2070,77** y estos coeficientes:

| Columna | Coeficiente | En lenguaje de negocio |
|---|---|---|
| `temperatura_c` | 152,56 | Cada grado más, unas 153 bicicletas más, con lo demás igual |
| `humedad` | -16,75 | Cada punto más de humedad, unas 17 menos |
| `viento` | -51,17 | Cada punto más de viento, unas 51 menos |
| `es_2012` | 1959,94 | Un día de 2012 tiene unas 1960 más que uno parecido de 2011: el uso del sistema creció |
| `dia_laboral` | 145,40 | Un día laboral tiene unas 145 más: poca diferencia |
| `lluvia` | -1239,71 | Un día con lluvia o nieve, unas 1240 menos |

**"Con lo demás igual"** es la clave de la regresión múltiple: cada coeficiente es el efecto de esa columna **manteniendo fijas las otras**. Por eso el de la temperatura bajó un poco (de 160 a 153): parte de lo que antes se le atribuía era del año o de la lluvia.

**Cuidado:** un coeficiente grande no significa una columna más importante. Depende de las unidades: un punto de humedad no es comparable con pasar de 2011 a 2012.

In [ ]:
pred_multiple = multiple.predict(prueba[columnas])

print("MAE: ", round(mean_absolute_error(y_test, pred_multiple), 1))
print("RMSE:", round(mean_squared_error(y_test, pred_multiple) ** 0.5, 1))
print("R²:  ", round(r2_score(y_test, pred_multiple), 3))

**Debes ver:** MAE **733,4**, RMSE **919,1** y R² **0,789**. Junta los tres resultados en una tabla:

In [ ]:
comparacion = pd.DataFrame({
    "modelo": ["Línea base (promedio)", "Simple (temperatura)", "Múltiple (6 columnas)"],
    "MAE": [mean_absolute_error(y_test, pred_base),
            mean_absolute_error(y_test, pred_simple),
            mean_absolute_error(y_test, pred_multiple)],
    "R2": [r2_score(y_test, pred_base),
           r2_score(y_test, pred_simple),
           r2_score(y_test, pred_multiple)],
})
comparacion.round(3)

El error promedio baja de unas 1700 bicicletas a unas 730: **menos de la mitad**. Ese es el aporte del modelo, medido con días que no vio.

El R² de la línea base sale un poco negativo (-0,020): el promedio del entrenamiento no es exactamente el de la prueba. Un R² cercano a 0, o negativo, significa "no mejor que el promedio".

## 8. Predecir un día nuevo

Un día laboral de 2012, a 25 grados, con humedad de 60, viento de 12 y sin lluvia. ¿Cuántas bicicletas preparar? ¿Y si el pronóstico cambia a lluvia?

In [ ]:
dia_nuevo = pd.DataFrame({
    "temperatura_c": [25, 25],
    "humedad": [60, 60],
    "viento": [12, 12],
    "es_2012": [1, 1],
    "dia_laboral": [1, 1],
    "lluvia": [0, 1],
})

multiple.predict(dia_nuevo).round(0)

**Debes ver:** unas **6371** bicicletas sin lluvia y unas **5131** con lluvia. Recuerda que el modelo se equivoca, en promedio, en unas 730: la predicción es una guía para planear, no una cifra exacta.

## 9. Los límites del modelo

**Extrapolar** es pedirle al modelo una predicción fuera del rango de los datos con que aprendió. Los días de entrenamiento van de 2,4 a 35,3 grados. Mira qué dice el modelo simple fuera de ese rango:

In [ ]:
extremos = pd.DataFrame({"temperatura_c": [-10, 45]})
simple.predict(extremos).round(0)

**Debes ver:** unas **-331** bicicletas a -10 grados (¡un número negativo de alquileres!) y **8488** a 45 grados, aunque en el paso 6 viste que con calor fuerte los alquileres **bajan**. La recta sigue subiendo para siempre; la realidad no. **Nunca uses el modelo lejos de los datos con que aprendió.**

Ahora, los días de prueba en que más se equivocó el modelo múltiple:

In [ ]:
prueba = prueba.copy()
prueba["prediccion"] = pred_multiple.round(0)
prueba["error"] = prueba["alquileres"] - prueba["prediccion"]

prueba.sort_values("error")[["fecha", "clima", "temperatura_c", "alquileres", "prediccion", "error"]].head(4)

**Debes ver** en primer lugar el **29 de octubre de 2012**: el modelo predijo unas 2980 bicicletas y se alquilaron 22. Ese día llegó el huracán Sandy a la costa este de Estados Unidos: ningún dato del clima diario anticipa una ciudad cerrada. Le sigue el 29 de junio de 2012, a 34,2 grados: el modelo predijo unas 8006 y se alquilaron 5463. Otra vez, el calor fuerte que la recta no captura.

Un buen análisis no esconde estos casos: los explica. Son la parte de la realidad que el modelo no ve.

## 10. Taller: explica tu modelo en lenguaje de negocio

Copia la ficha en una celda de texto y complétala con tus resultados. Es lo que entregas al final de la clase.

| Pregunta | Tu respuesta |
|---|---|
| ¿Qué decisión apoya el modelo? | |
| ¿Cuánto mejora a la línea base? (MAE de la base y del modelo) | |
| ¿Qué dos coeficientes le contarías a la gerencia, y cómo los dirías? | |
| ¿En qué días se equivoca más, y por qué? | |
| ¿Qué no debe hacerse con este modelo? | |

**Retos opcionales:**

1. Agrega `sensacion_termica_c` al modelo múltiple. ¿Qué les pasa a los coeficientes de la temperatura y de la sensación térmica? ¿Mejora el MAE? Explica qué tiene que ver con la pregunta 3 de la selección de características de la Clase 19.
2. Quita `es_2012`. ¿Cuánto empeora el modelo? ¿Qué significa para una empresa que el año sea tan importante?